# MNIST Classification with OpenEye - Quick Start

This notebook provides a streamlined workflow for training an MNIST model and deploying it on the OpenEye accelerator.

For a more detailed tutorial with multi-framework support, see:
- [mnist_tensorflow.ipynb](mnist_tensorflow.ipynb) - Complete TensorFlow/Keras workflow
- [mnist_pytorch.ipynb](mnist_pytorch.ipynb) - PyTorch workflow
- [mnist_onnx.ipynb](mnist_onnx.ipynb) - ONNX workflow
- [unified_model_loader_demo.ipynb](unified_model_loader_demo.ipynb) - Multi-framework demo

## Quick Start Steps
1. Train CNN model with Keras
2. Quantize with TFLite
3. Simulate on OpenEye RTL

## Step 1: Train Model

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical

print(f"TensorFlow version: {tf.__version__}")

# Load and preprocess the MNIST dataset
(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train = x_train.reshape((x_train.shape[0], 28, 28, 1)).astype('float32') / 255
x_test = x_test.reshape((x_test.shape[0], 28, 28, 1)).astype('float32') / 255
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

print(f"Training data: {x_train.shape}")
print(f"Test data: {x_test.shape}")

In [ ]:
# Build the CNN model
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(28, 28, 1)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
])

# Compile the model
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

print("Model architecture:")
model.summary()

In [ ]:
# Train the model
print("\nTraining...")
history = model.fit(x_train, y_train, epochs=5, batch_size=64, validation_split=0.1, verbose=1)

# Evaluate the model
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"\nTest accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")

## Step 2: Quantize Model

In [ ]:
# TensorFlow Lite conversion with INT8 quantization
rep_ds_size = 100
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.target_spec.supported_types = [tf.int8]

def representative_dataset_gen():
    """Representative dataset generator for Post-Training Quantization"""
    for i in range(rep_ds_size):
        sample = np.expand_dims(x_train[i], axis=0).astype(np.float32)
        yield [sample]

converter.representative_dataset = representative_dataset_gen
tflite_model = converter.convert()

# Save the quantized model
model_name = "mnist_quantized_model"
model_path = model_name + '.tflite'
with open(model_path, 'wb') as f:
    f.write(tflite_model)

print(f"\n✅ Quantized model saved as: {model_path}")

# Show size comparison
import os
tflite_size = os.path.getsize(model_path) / 1024
print(f"   Model size: {tflite_size:.2f} KB")

## Step 3: Test Quantized Model

In [ ]:
# Test the quantized model
interpreter = tf.lite.Interpreter(model_content=tflite_model)
interpreter.allocate_tensors()

# Get input and output details
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("Quantized Model Details:")
print(f"  Input shape: {input_details[0]['shape']}")
print(f"  Input type: {input_details[0]['dtype']}")
print(f"  Output shape: {output_details[0]['shape']}")
print(f"  Output type: {output_details[0]['dtype']}")

# Test with a sample
test_sample = x_test[0:1].astype(np.float32)
interpreter.set_tensor(input_details[0]['index'], test_sample)
interpreter.invoke()
tflite_result = interpreter.get_tensor(output_details[0]['index'])

original_pred = np.argmax(model.predict(test_sample, verbose=0))
quantized_pred = np.argmax(tflite_result)

print(f"\nPrediction comparison:")
print(f"  Original model:  {original_pred}")
print(f"  Quantized model: {quantized_pred}")
print(f"  Match: {'✅' if original_pred == quantized_pred else '❌'}")

## Step 4: Load with OpenEye (Optional)

Use OpenEye's unified model loader to inspect the model:

In [ ]:
import sys
import os

# Add OpenEye source to path
openeye_base = os.path.abspath(os.path.join(os.pardir, os.pardir))
sys.path.insert(0, os.path.join(openeye_base, "src"))

try:
    from open_eye.model_loader import load_model
    
    # Load model with unified loader
    openeye_model = load_model(model_path)
    
    print("\n" + "="*60)
    print("OpenEye Model Properties:")
    print("="*60)
    print(f"Framework:    {openeye_model.framework}")
    print(f"Layers:       {openeye_model.num_layers}")
    print(f"Input shape:  {openeye_model.input_shape}")
    print(f"Output shape: {openeye_model.output_shape}")
    print(f"Quantized:    {openeye_model.is_quantized}")
    
    print("\nLayer structure:")
    for i, layer_type in enumerate(openeye_model.get_layer_types()):
        print(f"  {i}: {layer_type}")
    
    print("\n✅ Model successfully loaded with OpenEye!")
    
except ImportError as e:
    print(f"⚠️  OpenEye model loader not available: {e}")
    print("   This is optional - you can still run the simulation below.")

## Step 5: Simulate on OpenEye RTL

Run cocotb simulation on the OpenEye RTL design:

In [ ]:
import pytest
import cocotb_test
import cocotb_test.simulator

# Setup paths
tests_dir = os.path.abspath(os.path.join(openeye_base, "test"))
tb_dir = os.path.join(tests_dir, "cocotb_fpga")
hdl_dir = os.path.join(openeye_base, "hdl")

sys.path.append(tests_dir)

import open_eye.test_utils_main as tu
import open_eye.open_eye_parameters as oe_params
import open_eye.vh_file_creator as vh_file_creator

print(f"Test directory: {tests_dir}")
print(f"HDL directory:  {hdl_dir}")

In [ ]:
# Define simulation parameters
clk_cycle = 20
clk_cycle_unit = "ns"

clk_delay_in = 100
clk_delay_unit_in = "ps"

clk_delay_out = 100
clk_delay_unit_out = "ps"

dut = 'OpenEye_FPGA'
module = 'OpenEye_FPGA_tb'
toplevel = dut

print(f"DUT: {dut}")
print(f"Testbench module: {module}")

In [ ]:
# Get Verilog sources
verilog_sources = tu.get_verilog_sources(hdl_dir)
target_dir = os.path.join(tests_dir, 'simulation/' + model_name)

print(f"Found {len(verilog_sources)} Verilog source files")
print(f"Simulation directory: {target_dir}")

In [ ]:
# Create hardware parameters file
import nest_asyncio
nest_asyncio.apply()

oep = oe_params.OpenEyeParameters()
vh_file_creator.create_vh_file(oep)

print("✅ Hardware parameters file created")

In [ ]:
# Run cocotb simulation
print("\n" + "="*60)
print("Starting cocotb simulation...")
print("="*60 + "\n")

results = cocotb_test.simulator.run(
    python_search=[tb_dir],
    verilog_sources=verilog_sources,
    toplevel=toplevel,
    module=module,
    sim_build=target_dir,
    testcase='model_test',
    force_compile=False,
    waves=True,
    includes=[hdl_dir],
    simulator="icarus",
    extra_env={
        "CLOCK_LEN": str(clk_cycle),
        "CLOCK_UNIT": clk_cycle_unit,
        "CLOCK_DELAY_INPUT": str(clk_delay_in),
        "CLOCK_DELAY_UNIT_INPUT": clk_delay_unit_in,
        "CLOCK_DELAY_OUTPUT": str(clk_delay_out),
        "CLOCK_DELAY_UNIT_OUTPUT": clk_delay_unit_out,
        "MODEL_PATH": model_path
    }
)

print("\n✅ Simulation completed!")

## Summary

This notebook demonstrated a quick MNIST workflow:

✅ **Step 1:** Trained CNN model with Keras  
✅ **Step 2:** Quantized to INT8 with TFLite  
✅ **Step 3:** Tested quantized model  
✅ **Step 4:** Loaded with OpenEye unified loader (optional)  
✅ **Step 5:** Simulated on OpenEye RTL  

**For more detailed workflows, see:**
- [mnist_tensorflow.ipynb](mnist_tensorflow.ipynb) - Complete TensorFlow/Keras tutorial
- [mnist_pytorch.ipynb](mnist_pytorch.ipynb) - PyTorch workflow
- [mnist_onnx.ipynb](mnist_onnx.ipynb) - ONNX workflow  
- [unified_model_loader_demo.ipynb](unified_model_loader_demo.ipynb) - Multi-framework comparison

**Next Steps:**
1. Analyze simulation waveforms
2. Try different model architectures
3. Experiment with other frameworks
4. Deploy on FPGA hardware